# Avito AntiBot — финальное решение

## Краткое описание

Задача решается как бинарная классификация на уровне `cookie_id`.
Все признаки строятся только по событиям, которые доступны внутри окна наблюдения:

`window_start_ts <= event_ts < window_end_ts`.

### Валидация
Так как `test` расположен позже `train` по времени, используется временной holdout:
объекты до 17 апреля идут в обучение, 17–19 апреля — в validation.

### Что пробовалось
1. **Baseline:** RandomForest на двух простых признаках — количестве событий и количестве уникальных объявлений.
2. **Основная модель:** CatBoost на расширенных поведенческих агрегатах.
3. Дополнительно проверялись feature selection и ансамбли с другими бустингами, но они не давали устойчивого выигрыша на временных срезах, поэтому финальная версия оставлена более простой.

### Основные группы признаков
- возраст cookie и параметры окна;
- интенсивность и длительность активности;
- интервалы между соседними событиями и их квантили;
- доля очень быстрых действий;
- число уникальных объявлений, категорий, локаций и поисковых запросов;
- доли разных типов событий;
- platform/User-Agent;
- search_page;
- pointer_x/pointer_y;
- концентрация действий на одном объекте;
- количество полных дубликатов событий.

Случайность фиксирована через `random_state/random_seed`.

## 1. Импорты и загрузка данных

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path("data") if Path("data").exists() else Path(".")

train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
test = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)

events_path = DATA_DIR / "events.csv.gz"
if not events_path.exists():
    events_path = DATA_DIR / "events.csv"

events = pd.read_csv(events_path, parse_dates=["event_ts"])

RANDOM_STATE = 43
EVENT_NAMES = sorted(events["event_name"].dropna().unique().tolist())

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)
print("target mean:", round(train["target"].mean(), 4))

## 2. Официальная логика метрики

In [ ]:

def pr_curve(y_true, score):
    """Precision/recall на границах групп одинакового score."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)

    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")

    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.array([]), np.array([])

    order = np.argsort(-score, kind="mergesort")
    y = y_true[order]
    s = score[order]

    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]

    return tp[ends] / k[ends], tp[ends] / n_pos


def precision_at_recall(y_true, score, recall=0.70):
    """Максимальный precision среди порогов с recall >= 0.70."""
    prec, rec = pr_curve(y_true, score)
    if len(prec) == 0:
        return float("nan")
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0


## 3. Подготовка событий и признаков

In [ ]:

def entropy_from_counts(arr):
    arr = np.asarray(arr, dtype=float)
    total = arr.sum()
    if total <= 0:
        return 0.0
    p = arr[arr > 0] / total
    return float(-(p * np.log(p)).sum())


def normalize_platform(s):
    """Нормализует разные написания одной платформы."""
    s = s.fillna("missing").astype(str).str.lower()
    return s.replace({
        "web": "web",
        "desktop": "web",
        "android": "android",
        "ios": "ios",
        "iphone": "ios",
    })


def events_in_window(events, meta):
    """
    Оставляет только события, доступные к моменту окончания окна:
    window_start_ts <= event_ts < window_end_ts.
    """
    ev = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
    )
    mask = (
        (ev["event_ts"] >= ev["window_start_ts"])
        & (ev["event_ts"] < ev["window_end_ts"])
    )
    return ev.loc[mask].drop(columns=["window_start_ts", "window_end_ts"])


def build_features(ev, meta, event_names):
    """
    Одна строка признаков на cookie_id.

    Группы признаков:
    - возраст cookie и параметры окна;
    - интенсивность и длительность активности;
    - интервалы между событиями;
    - разнообразие объявлений/категорий/локаций/запросов;
    - доли типов событий;
    - platform/User-Agent;
    - search_page;
    - pointer_x/pointer_y;
    - концентрация активности на одном объекте.
    """
    raw_counts = ev.groupby("cookie_id").size().rename("raw_n_events")

    # Полные дубликаты считаем отдельным признаком, затем удаляем,
    # чтобы они не удваивали остальные агрегаты.
    d = ev.drop_duplicates().copy()
    d["platform_norm"] = normalize_platform(d["platform"])
    d = d.sort_values(["cookie_id", "event_ts"], kind="mergesort")

    idx = pd.Index(meta["cookie_id"], name="cookie_id")
    meta_i = meta.set_index("cookie_id")
    f = pd.DataFrame(index=idx)
    g = d.groupby("cookie_id", sort=False)

    # Метаданные cookie
    f["cookie_age_hours"] = (
        (meta_i["window_start_ts"] - meta_i["cookie_created_at"])
        .dt.total_seconds()
        .div(3600)
        .reindex(idx)
    )
    f["window_hours"] = (
        (meta_i["window_end_ts"] - meta_i["window_start_ts"])
        .dt.total_seconds()
        .div(3600)
        .reindex(idx)
    )
    f["window_start_dow"] = meta_i["window_start_ts"].dt.dayofweek.reindex(idx)
    f["window_start_day"] = meta_i["window_start_ts"].dt.day.reindex(idx)

    # Общее число событий и полные дубликаты
    n_events = g.size()
    f["raw_n_events"] = raw_counts.reindex(idx).fillna(0)
    f["n_events"] = n_events.reindex(idx).fillna(0)
    f["n_duplicate_rows"] = (raw_counts - n_events).reindex(idx).fillna(0)
    f["duplicate_fraction"] = (
        f["n_duplicate_rows"] / f["raw_n_events"].replace(0, np.nan)
    ).fillna(0)

    # Длительность активности
    tmin = g["event_ts"].min().reindex(idx)
    tmax = g["event_ts"].max().reindex(idx)
    f["activity_span_sec"] = (tmax - tmin).dt.total_seconds().fillna(0)
    f["events_per_active_hour"] = np.where(
        f["activity_span_sec"] > 0,
        f["n_events"] / (f["activity_span_sec"] / 3600),
        f["n_events"],
    )
    f["first_event_offset_sec"] = (
        tmin - meta_i["window_start_ts"].reindex(idx)
    ).dt.total_seconds().fillna(0)
    f["last_event_before_end_sec"] = (
        meta_i["window_end_ts"].reindex(idx) - tmax
    ).dt.total_seconds().fillna(0)

    # Разнообразие значений
    nunique_cols = [
        "eid", "event_name", "platform_norm", "user_agent",
        "item_id", "item_category", "item_location",
        "seller_type", "search_query", "search_page",
    ]
    for col in nunique_cols:
        f[f"{col}_nunique"] = g[col].nunique(dropna=True).reindex(idx).fillna(0)

    for col in ["item_id", "item_category", "item_location", "search_query", "user_agent"]:
        f[f"{col}_unique_ratio"] = (
            f[f"{col}_nunique"] / f["n_events"].replace(0, np.nan)
        ).fillna(0)

    # Доли заполненных полей
    for col in [
        "item_id", "item_category", "item_location", "seller_type",
        "search_query", "search_page", "pointer_x", "pointer_y", "user_agent",
    ]:
        nonnull = g[col].count().reindex(idx).fillna(0)
        f[f"{col}_nonnull_frac"] = (
            nonnull / f["n_events"].replace(0, np.nan)
        ).fillna(0)

    # Типы событий
    event_counts = pd.crosstab(d["cookie_id"], d["event_name"])
    for name in event_names:
        vals = event_counts[name] if name in event_counts.columns else pd.Series(dtype=float)
        f[f"event_{name}_count"] = vals.reindex(idx).fillna(0)
        f[f"event_{name}_ratio"] = (
            f[f"event_{name}_count"] / f["n_events"].replace(0, np.nan)
        ).fillna(0)

    # Платформы
    platform_counts = pd.crosstab(d["cookie_id"], d["platform_norm"])
    for platform in ["web", "android", "ios", "missing"]:
        vals = platform_counts[platform] if platform in platform_counts.columns else pd.Series(dtype=float)
        f[f"platform_{platform}_count"] = vals.reindex(idx).fillna(0)
        f[f"platform_{platform}_ratio"] = (
            f[f"platform_{platform}_count"] / f["n_events"].replace(0, np.nan)
        ).fillna(0)

    d["platform_switch"] = (
        d["platform_norm"].ne(d.groupby("cookie_id")["platform_norm"].shift()).astype(int)
    )
    switches = d.groupby("cookie_id")["platform_switch"].sum() - 1
    f["platform_switches"] = switches.clip(lower=0).reindex(idx).fillna(0)

    # Номер страницы поиска
    search_page = d.groupby("cookie_id")["search_page"]
    for stat, series in [
        ("mean", search_page.mean()),
        ("std", search_page.std()),
        ("min", search_page.min()),
        ("max", search_page.max()),
        ("median", search_page.median()),
    ]:
        f[f"search_page_{stat}"] = series.reindex(idx).fillna(0)

    # Координаты курсора
    for col in ["pointer_x", "pointer_y"]:
        gg = d.groupby("cookie_id")[col]
        for stat, series in [
            ("mean", gg.mean()),
            ("std", gg.std()),
            ("min", gg.min()),
            ("max", gg.max()),
            ("median", gg.median()),
        ]:
            f[f"{col}_{stat}"] = series.reindex(idx).fillna(0)
        f[f"{col}_nunique"] = gg.nunique().reindex(idx).fillna(0)

    # User-Agent: используем общие свойства строки, а не строку целиком
    ua = d["user_agent"].fillna("").astype(str)
    d["ua_len"] = ua.str.len()
    patterns = {
        "ua_python": r"python|requests|aiohttp|urllib",
        "ua_headless": r"headless",
        "ua_curl_wget": r"curl|wget",
        "ua_bot_word": r"bot|spider|crawler|scrapy",
        "ua_mobile": r"mobile|android|iphone",
        "ua_chrome": r"chrome|chromium",
        "ua_firefox": r"firefox",
        "ua_safari": r"safari",
    }
    ua_low = ua.str.lower()
    for name, pattern in patterns.items():
        d[name] = ua_low.str.contains(pattern, regex=True, na=False).astype(int)

    gu = d.groupby("cookie_id")
    f["ua_len_mean"] = gu["ua_len"].mean().reindex(idx).fillna(0)
    f["ua_len_std"] = gu["ua_len"].std().reindex(idx).fillna(0)
    f["ua_len_max"] = gu["ua_len"].max().reindex(idx).fillna(0)
    for name in patterns:
        f[f"{name}_frac"] = gu[name].mean().reindex(idx).fillna(0)

    # Интервалы между соседними событиями
    d["dt"] = d.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
    gd = d.groupby("cookie_id")["dt"]

    quantiles = gd.quantile([0.10, 0.25, 0.50, 0.75, 0.90]).unstack()
    if len(quantiles):
        quantiles.columns = [f"dt_q{int(q * 100)}" for q in quantiles.columns]
        for col in quantiles.columns:
            f[col] = quantiles[col].reindex(idx).fillna(0)

    for stat, series in [
        ("mean", gd.mean()),
        ("std", gd.std()),
        ("min", gd.min()),
        ("max", gd.max()),
    ]:
        f[f"dt_{stat}"] = series.reindex(idx).fillna(0)

    interval_count = (f["n_events"] - 1).clip(lower=0)
    for sec in [0, 1, 2, 5, 10, 30, 60, 300, 900, 3600]:
        count = d["dt"].le(sec).groupby(d["cookie_id"]).sum().reindex(idx).fillna(0)
        f[f"dt_le_{sec}_frac"] = (
            count / interval_count.replace(0, np.nan)
        ).fillna(0)

    f["dt_cv"] = (
        f["dt_std"] / f["dt_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0)

    f["dt_burstiness"] = (
        (f["dt_std"] - f["dt_mean"])
        / (f["dt_std"] + f["dt_mean"]).replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0)

    # Активность по времени суток
    d["hour"] = d["event_ts"].dt.hour
    d["minute_of_day"] = d["event_ts"].dt.hour * 60 + d["event_ts"].dt.minute
    f["active_hours"] = d.groupby("cookie_id")["hour"].nunique().reindex(idx).fillna(0)
    f["active_minutes"] = d.groupby("cookie_id")["minute_of_day"].nunique().reindex(idx).fillna(0)
    f["events_per_active_minute"] = (
        f["n_events"] / f["active_minutes"].replace(0, np.nan)
    ).fillna(0)

    hour_counts = d.groupby(["cookie_id", "hour"]).size().rename("cnt").reset_index()
    hour_entropy = hour_counts.groupby("cookie_id")["cnt"].apply(
        lambda x: entropy_from_counts(x.values)
    )
    f["hour_entropy"] = hour_entropy.reindex(idx).fillna(0)

    # Повтор типов событий
    d["same_event_prev"] = (
        d["event_name"].eq(d.groupby("cookie_id")["event_name"].shift()).astype(int)
    )
    f["same_event_prev_frac"] = (
        d.groupby("cookie_id")["same_event_prev"].mean().reindex(idx).fillna(0)
    )

    f["item_view_unique_ratio"] = (
        f["item_id_nunique"] / f["event_item_view_count"].replace(0, np.nan)
    ).clip(upper=10).fillna(0)

    # Концентрация активности
    for col in ["item_id", "item_category", "item_location", "search_query", "event_name"]:
        dd = d.dropna(subset=[col])
        if len(dd):
            counts = dd.groupby(["cookie_id", col]).size().rename("cnt")
            top = counts.groupby(level=0).max()
            total = counts.groupby(level=0).sum()
            f[f"{col}_top_share"] = (top / total).reindex(idx).fillna(0)
        else:
            f[f"{col}_top_share"] = 0.0

    return (
        f.replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .reset_index()
    )


In [ ]:
ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)

print("events in train windows:", len(ev_tr))
print("events in test windows:", len(ev_te))

Xtr = build_features(ev_tr, train, EVENT_NAMES)
Xte = build_features(ev_te, test, EVENT_NAMES)

feature_cols = [c for c in Xtr.columns if c != "cookie_id"]

y = (
    train.set_index("cookie_id")
    .loc[Xtr["cookie_id"], "target"]
    .to_numpy()
)

window_start = (
    train.set_index("cookie_id")
    .loc[Xtr["cookie_id"], "window_start_ts"]
)

print("number of features:", len(feature_cols))
Xtr.head()

## 4. Baseline

In [ ]:
def basic_features(ev, meta):
    g = ev.groupby("cookie_id")
    f = pd.DataFrame({
        "n_events": g.size(),
        "item_nunique": g["item_id"].nunique(),
    })
    return (
        meta[["cookie_id"]]
        .merge(f.reset_index(), on="cookie_id", how="left")
        .fillna(0)
    )

is_valid = window_start.ge("2026-04-17").to_numpy()

basic_tr = basic_features(ev_tr, train)

baseline = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=0,
)

baseline.fit(
    basic_tr.loc[~is_valid, ["n_events", "item_nunique"]],
    y[~is_valid],
)

p_baseline = baseline.predict_proba(
    basic_tr.loc[is_valid, ["n_events", "item_nunique"]]
)[:, 1]

print(
    "Baseline P@R>=0.70:",
    round(precision_at_recall(y[is_valid], p_baseline), 4),
)

## 5. CatBoost и временная валидация

In [ ]:
MODEL_PARAMS = dict(
    iterations=180,
    depth=8,
    learning_rate=0.035,
    l2_leaf_reg=7,
    random_strength=0.8,
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    verbose=False,
    thread_count=-1,
)

model = CatBoostClassifier(**MODEL_PARAMS)

model.fit(
    Xtr.loc[~is_valid, feature_cols],
    y[~is_valid],
)

p_valid = model.predict_proba(
    Xtr.loc[is_valid, feature_cols]
)[:, 1]

print(
    "P@R>=0.70:",
    round(precision_at_recall(y[is_valid], p_valid), 4),
)
print(
    "PR-AUC:",
    round(average_precision_score(y[is_valid], p_valid), 4),
)
print(
    "ROC-AUC:",
    round(roc_auc_score(y[is_valid], p_valid), 4),
)

## 6. Финальное обучение и `submission.csv`

После проверки схемы валидации модель обучается на всём `train`.
Порог выбирать не нужно: платформа сама перебирает пороги по `score`.

In [ ]:
final_model = CatBoostClassifier(**MODEL_PARAMS)
final_model.fit(Xtr[feature_cols], y)

test_score = final_model.predict_proba(
    Xte[feature_cols]
)[:, 1]

submission = pd.DataFrame({
    "cookie_id": Xte["cookie_id"],
    "score": test_score,
})

# Проверки формата
assert len(submission) == len(test)
assert submission["cookie_id"].is_unique
assert submission["cookie_id"].notna().all()
assert submission["score"].between(0, 1).all()
assert set(submission["cookie_id"]) == set(test["cookie_id"])

submission.to_csv("submission.csv", index=False)

print("submission shape:", submission.shape)
submission.head()

## Ограничения решения

- Временная валидация ближе к скрытому тесту, но доступная история короткая, поэтому возможен temporal drift.
- Модель использует агрегаты по событиям и не моделирует длинные последовательности напрямую.
- Категориальные строки не подаются модели как необработанные идентификаторы; используются статистики и агрегаты.
- Внешние API, LLM, ручная разметка test и хардкод `cookie_id` не используются.